# PaleoWave — 03 ML Predictive Locality Model
Trains a Random Forest on PBDB terrain features, generates a probability
surface across the full study area using windowed reads, and extracts
ranked candidate localities cross-referenced with Nevada Triassic geology.

**Outputs:**
- `data/model/model_rf.pkl` — trained Random Forest
- `data/model/prediction_surface.tif` — fossil probability raster
- `data/model/prediction_map.png` — visualization
- `data/model/top_candidates.geojson` — top 50 sites
- `data/model/priority_targets.csv` — ranked with geology cross-reference
- `data/model/priority_targets.geojson` — field-ready GeoJSON

> **Key lesson:** The merged DEM is 700M pixels. NEVER load it fully into
> memory. Use `rasterio` windowed reads and `src.sample()` throughout.

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import Point
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

DATA_DIR    = Path('../data')
TERRAIN_DIR = DATA_DIR / 'terrain'
MODEL_DIR   = DATA_DIR / 'model'
GEO_DIR     = DATA_DIR / 'geology'
MODEL_DIR.mkdir(exist_ok=True)

FEATURE_COLS = ['elevation_m', 'slope_deg', 'aspect_deg', 'tri']
RASTER_PATHS = {
    'elevation_m': DATA_DIR / 'dem/dem_merged.tif',
    'slope_deg':   TERRAIN_DIR / 'slope.tif',
    'aspect_deg':  TERRAIN_DIR / 'aspect.tif',
    'tri':         TERRAIN_DIR / 'ruggedness.tif',
}
print('Ready.')

## 2. Load Positive Examples

In [ ]:
positives = pd.read_csv(DATA_DIR / 'features_pbdb_terrain.csv')
positives = positives.dropna(subset=FEATURE_COLS)
positives['label'] = 1
print(f'Positive examples: {len(positives)}')
print(positives[FEATURE_COLS].describe().round(2))

## 3. Generate Background Points
Random coords sampled from raster extent via `src.sample()` — no full load.

In [ ]:
N_BACKGROUND = len(positives) * 10
np.random.seed(42)

with rasterio.open(RASTER_PATHS['slope_deg']) as src:
    h, w = src.height, src.width
    transform = src.transform
    crs = src.crs

print(f'Raster: {w}x{h} = {w*h:,} pixels')

# Random pixel indices -> coordinates
bg_rows = np.random.randint(0, h, size=N_BACKGROUND*3)
bg_cols = np.random.randint(0, w, size=N_BACKGROUND*3)
bg_lons = transform.c + bg_cols * transform.a + transform.a/2
bg_lats = transform.f + bg_rows * transform.e + transform.e/2
coords  = list(zip(bg_lons, bg_lats))

bg = pd.DataFrame({'longitude': bg_lons, 'latitude': bg_lats})
for col, path in RASTER_PATHS.items():
    with rasterio.open(path) as src:
        bg[col] = [v[0] for v in src.sample(coords)]

bg = bg[bg['elevation_m'] > -9999].copy()
bg['aspect_deg'] = bg['aspect_deg'] % 360
bg = bg.dropna(subset=FEATURE_COLS).head(N_BACKGROUND)
bg['label'] = 0
print(f'Background points: {len(bg)}')

## 4. Build Training Set & Train Model

In [ ]:
df = pd.concat([positives[FEATURE_COLS+['label']], bg[FEATURE_COLS+['label']]], ignore_index=True)
X  = df[FEATURE_COLS].values
y  = df['label'].values
print(f'Training: {len(df)} samples ({y.sum()} pos / {(y==0).sum()} bg)')

rf = RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=2,
                             class_weight='balanced', random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=cv, scoring='roc_auc')
print(f'CV AUC: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print(f'Per-fold: {["{:.3f}".format(s) for s in cv_scores]}')

rf.fit(X, y)

print('\nFeature importances:')
for feat, imp in sorted(zip(FEATURE_COLS, rf.feature_importances_), key=lambda x: -x[1]):
    print(f'  {feat:15s}: {imp:.3f}')

with open(MODEL_DIR/'model_rf.pkl','wb') as f:
    pickle.dump(rf, f)
print('Model saved.')

## 5. Generate Prediction Surface
Windowed reads — 500 rows at a time. Never loads full raster.

In [ ]:
pred_path = MODEL_DIR / 'prediction_surface.tif'
pred_meta = {'driver':'GTiff','dtype':'float32','count':1,'height':h,'width':w,
             'transform':transform,'crs':crs,'nodata':-9999,'compress':'lzw'}
TILE_ROWS = 500

print(f'Writing prediction surface ({h} rows in {TILE_ROWS}-row chunks)...')
src_files = {col: rasterio.open(path) for col, path in RASTER_PATHS.items()}

with rasterio.open(pred_path,'w',**pred_meta) as dst:
    for row_start in range(0, h, TILE_ROWS):
        row_count = min(TILE_ROWS, h-row_start)
        win = Window(0, row_start, w, row_count)
        bands = {}
        for col, src in src_files.items():
            arr = src.read(1, window=win).astype(np.float32)
            if col == 'aspect_deg': arr = arr % 360
            bands[col] = arr.ravel()
        X_tile = np.stack([bands[c] for c in FEATURE_COLS], axis=1)
        valid  = (bands['elevation_m'] > -9999) & (bands['slope_deg'] > -9999)
        probs  = np.full(len(valid), -9999, dtype=np.float32)
        if valid.sum() > 0:
            probs[valid] = rf.predict_proba(X_tile[valid])[:,1]
        dst.write(probs.reshape(1, row_count, w), window=win)
        if row_start % 5000 == 0: print(f'  ... row {row_start}/{h}')

for src in src_files.values(): src.close()
print('Prediction surface complete.')

## 6. Extract Top Candidates & Visualize

In [ ]:
TOP_N = 50
step  = 3  # downsample for display and candidate extraction

with rasterio.open(pred_path) as src:
    transform_pred = src.transform
    out_h = src.height // step
    out_w = src.width  // step
    prob_ds = src.read(1, out_shape=(out_h, out_w), resampling=Resampling.average)

# Top candidates
valid_rows, valid_cols = np.where(prob_ds > 0)
valid_probs = prob_ds[valid_rows, valid_cols]
top_idx   = np.argsort(valid_probs)[-TOP_N:][::-1]
top_rows  = valid_rows[top_idx] * step
top_cols  = valid_cols[top_idx] * step
top_probs = valid_probs[top_idx]
top_lons  = transform_pred.c + top_cols * transform_pred.a
top_lats  = transform_pred.f + top_rows * transform_pred.e

candidates = pd.DataFrame({'rank': range(1,TOP_N+1), 'longitude': top_lons,
                            'latitude': top_lats, 'probability': top_probs})
gdf = gpd.GeoDataFrame(candidates,
      geometry=[Point(lon,lat) for lon,lat in zip(top_lons,top_lats)], crs='EPSG:4326')
gdf.to_file(MODEL_DIR/'top_candidates.geojson', driver='GeoJSON')
print('Top candidates saved.')
print(candidates.head(10).to_string(index=False))

## 7. Cross-Reference with Triassic Geology

In [ ]:
geology   = gpd.read_file(GEO_DIR/'triassic_marine.shp')
geology['geometry'] = geology.geometry.buffer(0)
cands_utm = gdf.to_crs('EPSG:32611')
geo_utm   = geology.to_crs('EPSG:32611')
geo_union = geo_utm.union_all()

print('Calculating distances to Triassic formations...')
cands_utm['dist_to_triassic_m'] = cands_utm.geometry.apply(
    lambda g: int(g.distance(geo_union)))

def get_nearest(geom):
    idx = geo_utm.geometry.distance(geom).idxmin()
    r = geo_utm.loc[idx]
    return r['ORIG_LABEL'], r['unit_name']

results = cands_utm.geometry.apply(get_nearest)
candidates['dist_to_triassic_m'] = cands_utm['dist_to_triassic_m'].values
candidates['formation_code']     = [r[0] for r in results]
candidates['formation_name']     = [r[1] for r in results]

def geo_bonus(dist):
    if   dist <=   100: return 0.20
    elif dist <=   500: return 0.15
    elif dist <=  2000: return 0.10
    elif dist <=  5000: return 0.05
    else:               return 0.00

candidates['geo_bonus']       = candidates['dist_to_triassic_m'].apply(geo_bonus)
candidates['composite_score'] = (candidates['probability'] + candidates['geo_bonus']).clip(0,1).round(4)
candidates['priority']        = candidates['composite_score'].rank(ascending=False, method='min').astype(int)
candidates = candidates.sort_values('composite_score', ascending=False).reset_index(drop=True)

candidates.to_csv(MODEL_DIR/'priority_targets.csv', index=False)
final_gdf = gpd.GeoDataFrame(candidates,
    geometry=[Point(lon,lat) for lon,lat in zip(candidates['longitude'],candidates['latitude'])],
    crs='EPSG:4326')
final_gdf.to_file(MODEL_DIR/'priority_targets.geojson', driver='GeoJSON')

print('Priority targets saved.')
print(candidates.head(10)[['priority','latitude','longitude','probability',
                             'dist_to_triassic_m','formation_code','composite_score']].to_string(index=False))

## 8. Prediction Map

In [ ]:
pw_cmap = LinearSegmentedColormap.from_list(
    'paleowave', ['#0d1117','#1a3a2a','#2d6a4f','#f0a500','#e63946'], N=256)

pbdb = pd.read_csv(DATA_DIR/'pbdb/pbdb_occurrences_clean.csv')
pbdb_nv = pbdb[pbdb['in_nevada']==True]

fig, axes = plt.subplots(1,2,figsize=(16,7))
fig.patch.set_facecolor('#0d1117')
hotspots = np.where(prob_ds >= np.nanpercentile(prob_ds[prob_ds>0],95), prob_ds, np.nan)

for ax, (data_arr, title) in zip(axes,[
    (prob_ds,'Fossil Probability Surface'),
    (hotspots,'Top 5% Hotspots')]):
    ax.set_facecolor('#0d1117')
    im = ax.imshow(data_arr, cmap=pw_cmap, vmin=0, vmax=1)
    plt.colorbar(im,ax=ax,fraction=0.046,pad=0.04).ax.yaxis.set_tick_params(color='white')
    ax.set_title(title, color='white', fontsize=12); ax.axis('off')
    px = ((pbdb_nv['longitude']-transform_pred.c)/(transform_pred.a*step)).values
    py = ((pbdb_nv['latitude'] -transform_pred.f)/(transform_pred.e*step)).values
    ax.scatter(px,py,s=50,color='white',edgecolors='black',linewidths=0.5,zorder=5,label='Known sites')
    cx = ((candidates['longitude']-transform_pred.c)/(transform_pred.a*step)).values
    cy = ((candidates['latitude'] -transform_pred.f)/(transform_pred.e*step)).values
    ax.scatter(cx,cy,s=40,color='#f0a500',marker='^',edgecolors='white',linewidths=0.3,zorder=6,label='Candidates')

axes[0].legend(facecolor='#1c1c1c',labelcolor='white',fontsize=8)
fig.suptitle('PaleoWave — Predicted Fossil Localities',color='white',fontsize=14,y=1.01)
plt.tight_layout()
map_path = MODEL_DIR/'prediction_map.png'
plt.savefig(map_path,dpi=150,bbox_inches='tight',facecolor='#0d1117')
plt.show()
print(f'Saved: {map_path}')

## Done

- `data/model/model_rf.pkl` — trained model (AUC 0.906)
- `data/model/prediction_surface.tif` — load in QGIS
- `data/model/prediction_map.png`
- `data/model/top_candidates.geojson`
- `data/model/priority_targets.csv` + `.geojson` — ranked field targets

**Next:** Load into QGIS, overlay geology, generate field report.